# Supplementary Analysis: Hybrid Retrieval with Yutu-Embedding

**Reproducible pipeline for cultural vs. commercial comment classification**

This notebook reproduces the analysis reported in the paper (Section 5, Findings B2).
It uses a hybrid retrieval approach combining:
1. **Semantic search** via Yutu-Embedding (Tencent, 2048-dim, mean pooling)
2. **Exhaustive keyword matching** with 554 cultural + 1162 commercial keywords

Data source: `sentiment_analysis_results_with_expanded_themes.csv` (334,548 raw comments, 323,742 after filtering len<3)

## Pipeline Overview
- **Step 1**: Encode all comments into 2048-dim embedding vectors (XPU-accelerated)
- **Step 2**: Hybrid retrieval + sentiment cross-tabulation
- **Step 3**: Per-topic cultural sentiment analysis (16 topics)
- **Step 4**: Three-position interpretive distribution (Appreciation / Suspicion / Entertainment)

**Requirements**: `transformers`, `sentence-transformers`, `torch` (with Intel XPU support), `numpy`, `csv`

In [ ]:
import os
import csv
import gc
import numpy as np
from collections import Counter, defaultdict

os.environ['HF_HOME'] = 'D:/model'

DATA_DIR = r'c:\Users\Restore Han\Desktop\WWM-CULTURE\data'
YUTU_REPO = 'tencent/Youtu-Embedding'

print('Imports done.')

## Step 1: Load and Encode Comments

Load comments from CSV, filter short comments (len < 3), encode using Yutu-Embedding with mean pooling.
This step produces `corpus_embeddings_v2.npy` (323,742 x 2048) and `corpus_meta_v2.npz`.

> **Note**: Encoding takes ~55 minutes on Intel Arc GPU. If pre-computed files exist, skip to Step 2.

In [ ]:
emb_path = os.path.join(DATA_DIR, 'corpus_embeddings_v2.npy')
meta_path = os.path.join(DATA_DIR, 'corpus_meta_v2.npz')

if os.path.exists(emb_path) and os.path.exists(meta_path):
    print('Pre-computed embeddings found. Loading...')
    corpus_embeddings = np.load(emb_path)
    meta = np.load(meta_path, allow_pickle=True)
    comments = meta['comments'].tolist()
    topics_list = meta['topics'].tolist()
    labels_list = meta['labels'].tolist()
    polarities = meta['polarities'].tolist()
    n = len(comments)
    print(f'Loaded: {n} comments, embeddings shape: {corpus_embeddings.shape}')
else:
    print('No pre-computed embeddings found. Run step1_encode_v2.py first.')
    print(f'  Command: python {os.path.join(DATA_DIR, "step1_encode_v2.py")}')

In [ ]:
print(f'Total comments: {n}')
print(f'Embedding dim: {corpus_embeddings.shape[1]}')
print(f'Polarity distribution: {Counter(polarities)}')
print(f'Label distribution (top 10): {Counter(labels_list).most_common(10)}')

## Step 2: Load Model for Query Encoding

Load Yutu-Embedding via SentenceTransformer (auto-applies mean pooling from `1_Pooling/config.json`).

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = 'xpu' if torch.xpu.is_available() else 'cpu'
print(f'Device: {device}')

model = SentenceTransformer(YUTU_REPO, device=device, trust_remote_code=True)
print(f'Model loaded. Pooling mode (mean): {model[1].pooling_mode_mean_tokens}')

## Step 3: Define Keywords and Retrieval Functions

Keywords are exhaustively expanded using the synonym table (`同义词表.txt`) and frequency analysis (`topic_stats.csv`).

In [ ]:
combined_texts = [comments[i] + ' ' + topics_list[i] for i in range(n)]
print(f'Built {n} combined text strings for keyword matching.')

In [ ]:
def semantic_search(query, threshold=0.65):
    q_emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    scores = np.dot(corpus_embeddings, q_emb.T).flatten()
    indices = np.where(scores >= threshold)[0]
    return indices

def keyword_match(keywords):
    mask = np.zeros(n, dtype=bool)
    for kw in keywords:
        for i in range(n):
            if not mask[i] and kw in combined_texts[i]:
                mask[i] = True
    return np.where(mask)[0]

def hybrid_retrieve(queries, keywords, sem_thresh=0.65):
    all_sem = set()
    for q in queries:
        idx = semantic_search(q, threshold=sem_thresh)
        all_sem.update(idx.tolist())
    kw_idx = keyword_match(keywords)
    combined = np.union1d(np.array(list(all_sem), dtype=np.int64), kw_idx.astype(np.int64))
    return combined

def report(idx_set, name):
    idx_set = np.asarray(idx_set, dtype=np.int64)
    ct = len(idx_set)
    pol = Counter(polarities[i] for i in idx_set)
    print(f'\n=== {name} (N={ct}) ===')
    for k, v in pol.most_common():
        print(f'  {k}: {v} ({v/ct*100:.1f}%)')
    return pol, ct

print('Retrieval functions defined.')

### 3a. Cultural Keywords (554 keywords)

Expanded from core cultural terms + synonym table + topic frequency analysis.

In [ ]:
cultural_kw = [
    '音乐', '琴', '乐器', '古曲', '文化', '历史', '传统', '武侠', '仙侠',
    '国风', '古风', '河西', '敦煌', '凉州', '秦腔', '百工', '墨家', '六艺',
    '诗词', '非遗', '民乐', '琵琶', '古琴', '箫', '笛', '石窟',
    '考据', '考究', '考据党', '考究党', '考据视频',
    '明制', '明制汉服', '明代服饰', '明代风格',
    '宋', '宋代', '宋廷', '宋辽', '辽', '辽代', '大辽',
    '契丹', '吐蕃',
    '归义军', '张议潮', '张淮深', '张淮鼎',
    '柴荣', '周世宗', '赵匡胤', '赵大', '赵二',
    '唐横刀', '唐刀',
    '五弦琵琶', '螺钿紫檀五弦琵琶', '小琵琶', '琵琶精',
    '开封', '汴京', '汴梁', '玉门',
    '凉州城', '凉州曲',
    '河西走廊', '河西五郡', '河西篇', '河西三部', '河西剧情', '河西篇章',
    '河西三部曲', '燕云河西', '河西暗涌', '河西暗线',
    '河西侠迹', '河西支线',
    '中渡桥',
    '少林', '少林寺',
    '仙剑奇侠传', '秦时明月', '盗墓笔记', '刺客信条',
    '李煜', '南烛公子', '小周后',
    '通宝', '铜钱', '唐钱案', '唐钱', '大历铜钱', '周元通宝', '宋元通宝', '月华通宝',
    '暗涌', '暗线', '暗流', '暗潮',
    '墨家机关', '墨家机关术', '墨山道', '墨门', '墨门三杰',
    '狂澜', '梨园', '九流门',
    '河南文旅', '坐忘道',
    '开放世界', '幽云十六州', '燕云十六州',
    '伊刀', '嗟夫刀法', '傲寒六诀',
    '曹敬观音', '盈盈', '寒姨', '洛神',
    '3A', 'AAA', '国产', '国产游戏', '国产武侠',
    '黑神话悟空', '黑神话', '黑悟空',
    '单机', '单机游戏', '单机内容', '单机玩家',
    '主线剧情', '主线任务', '剧情回溯', '剧情封神',
    'CG动画', 'CG',
    '蛇蝎心肠', '木鸢', '游侠',
    '江湖', '侠客', '武侠风', '武侠游戏',
    '轻功', '大轻功',
    'NPC', '世界观', '代入感', '沉浸感',
    '配音', 'BGM', '音效', '原声', '原创曲', '戏腔',
    '北宋', '唐代', '南唐', '五代十国',
    '金庸', '古剑', '荒野大镖客', '只狼', '对马岛之魂',
    '不羡仙', '梦中不羡仙', '火烧不羡仙', '神仙渡', '三更天',
    '剑', '枪', '扇', '伞', '陌刀', '横刀', '绳镖', '双刀',
    '投壶', '象棋', '叶子戏', '鲁班锁', '榫卯',
    '文化输出', '非遗文化',
    '安西军', '节度之死', '归唐', '收复',
    '江南', '长安', '秦川', '秦岭',
    '角色设计', '男角色', '女角色', '女侠',
    '叙事', '台词', '对话',
    '过场动画', '结算动画',
    '千夜', '江叔', '容鸢', '沈义伦', '田英', '郑鄂',
    '公子', '鬼公子', '白龙', '白玉京',
    '樊楼', '文津馆', '鬼市',
    '醉花阴', '锦鲤玉扇', '沧海龙吟',
    '梦境', '觉障林', '方外地', '孤云门', '天泉门',
    '青溪', '青丘',
    '浣溪沙', '抛春恨', '绕梁',
    '戏曲', '戏乐',
    '蜀戏春', '春以为期',
    '大白狼主', '白狼',
    '不良人', '四大名捕', '大理寺少卿',
    '麻布袋', '安琉璃', '鱼在藻',
    '战国袍', '明代', '明末',
    '打戏', '动作设计', '处决',
    '美术', '美术风格', '画风',
    '舞蹈',
    '真人', '演员', '导演', 'COS',
    '谭晶', '洛天依',
    'PV', '实机', '实机演示',
    '音乐会',
    '花灯', '七夕', '中秋', '春节',
    '人设', '文案',
    '意境', '氛围', '风景',
    'DLC',
    '种地', '种田', '采集', '钓鱼',
    '百面', '百面千相',
    '射雕', '流星蝴蝶剑', '卧龙',
    '封神',
]
print(f'Cultural keywords: {len(cultural_kw)}')

### 3b. Commercial Keywords (1162 keywords)

In [ ]:
commercial_kw = [
    'BUG', 'bug', '卡顿', '优化', '氪金', '充值', '付费', '手游', '移动', '手机',
    '抽卡', '外观', '皮肤', '时装', '庄园', '骗', '欺骗', '垃圾', '烂',
    '和鸣', '和鸣套', '和鸣商店', '和鸣池子', '和鸣套装', '和鸣外观',
    '袅袅', '袅袅之音',
    '家业', '家业系统', '家园', '家园系统', '家业种植', '家业养殖', '家业商店', '家业重建',
    '心法', '心法系统', '心法突破', '心法碎片', '心法书', '心法转换',
    '定音', '定音转律', '定音石', '转律', '转律石', '调律',
    '长鸣珠', '和鸣珠',
    '战令', '月卡', '互通',
    '百业', '百业战', '百业社', '百业商店',
    '网易', '网义', '牢易', '猪厂',
    '腾讯', '鹅厂', '某讯',
    '逆水寒', '原神', '永劫无间',
    '服务器', '运营', '策划', '画面', '性能', '帧率', '掉帧',
    '画质', '分辨率', '崩溃', '闪退', '黑屏', '延迟',
    '登录', '登录问题', '下载', '安装', '客户端',
    '定价', '价格', '买断制', '收费', '内购', '货币', '免费',
    '开服', '公测', '内测', '网游', '联机', '组队',
    '版本', '版本更新', '更新', '补丁',
    '流水', '营收', '盈利', '商业化', '经济系统',
    '外挂', '脚本', '封号', '举报', '封禁',
    '手柄', '键鼠', '键位',
    '退游', '退坑', '卸载',
    '捏脸', '染色', '套装', '等级', '战力', '属性', '词条', '强化',
    '商城', '商店', '抽奖', '保底', '爆率', '概率',
    '赛季', '排行', '竞技', '匹配', '排队',
    '内存', '显存', 'GPU', 'CPU', '显卡', '驱动',
    '移动端', '安卓', 'ios', 'PC端', 'PC', 'MAC', 'PS',
    '平台互通', '账号互通', '双端互通',
    '丁禹兮',
    '氪金玩家', '氪佬', '课金', '充钱',
    '修复', '穿模',
]
print(f'Commercial keywords: {len(commercial_kw)}')

## Step 4: Run Cultural vs. Commercial Retrieval

In [ ]:
print('Cultural retrieval (keyword matching)...')
cultural_idx = keyword_match(cultural_kw)
print(f'  Cultural keyword hits: {len(cultural_idx)}')

print('Commercial retrieval (keyword matching)...')
commercial_idx = keyword_match(commercial_kw)
print(f'  Commercial keyword hits: {len(commercial_idx)}')

overlap = len(np.intersect1d(cultural_idx, commercial_idx))
print(f'\nOverlap: {overlap} comments in both sets')

c_pol, c_ct = report(cultural_idx, 'Cultural Comments')
m_pol, m_ct = report(commercial_idx, 'Commercial Comments')

## Step 5: Per-Topic Cultural Sentiment (16 Topics)

In [ ]:
cultural_topic_keywords = {
    '音乐': ['音乐', '琴', '乐器', '古曲', '民乐', '琵琶', '古琴', '箫', '笛',
              '五弦琵琶', '小琵琶', '琵琶精', '弹琵琶', '和音', '助眠', '助眠曲'],
    '琴': ['琴', '古琴', '琵琶', '箫', '笛', '五弦琵琶', '小琵琶'],
    '国风': ['国风', '古风', '仙侠', '武侠'],
    '文化': ['文化', '考据', '考究', '非遗', '传统文化', '百工', '六艺'],
    '历史': ['历史', '宋', '辽', '契丹', '吐蕃', '归义军', '唐横刀', '通宝', '铜钱', '唐钱'],
    '河西': ['河西', '河西走廊', '河西五郡', '河西篇', '河西三部', '河西剧情',
              '中渡桥', '河西侠迹', '河西支线'],
    '敦煌': ['敦煌', '敦煌归义军', '沙洲归义军'],
    '凉州': ['凉州', '凉州城', '凉州曲', '凉州吐蕃'],
    '秦腔': ['秦腔'],
    '百工': ['百工', '六艺'],
    '墨家': ['墨家', '墨家机关', '墨山道', '墨门', '墨门三杰'],
    '六艺': ['六艺', '百工'],
    '武侠': ['武侠', '唐横刀', '少林', '傲寒六诀', '嗟夫刀法', '止戈', '连招'],
    '古风': ['古风', '国风', '仙侠'],
    '传统': ['传统', '传统文化', '非遗', '明制', '明制汉服'],
    '非遗': ['非遗', '文化遗产', '传统文化', '百工', '六艺'],
}

topic_stats = defaultdict(lambda: {'pos': 0, 'neg': 0, 'neu': 0, 'total': 0})
pol_map = {'积极': 'pos', '消极': 'neg', '中性': 'neu'}

for topic_name, kws in cultural_topic_keywords.items():
    kw_idx = keyword_match(kws)
    for i in kw_idx:
        topic_stats[topic_name]['total'] += 1
        topic_stats[topic_name][pol_map[polarities[i]]] += 1

print('=== Cultural Topic x Sentiment ===')
header = '{:<10} {:>8} {:>10} {:>10} {:>10} {:>8} {:>8}'.format(
    'Topic', 'Total', 'Positive', 'Negative', 'Neutral', 'Pos%', 'Neg%')
print(header)
print('-' * 70)
for tname in sorted(topic_stats.keys(), key=lambda x: topic_stats[x]['total'], reverse=True):
    s = topic_stats[tname]
    if s['total'] > 50:
        print('{:<10} {:>8} {:>10} {:>10} {:>10} {:>7.1f}% {:>7.1f}%'.format(
            tname, s['total'], s['pos'], s['neg'], s['neu'],
            s['pos']/s['total']*100, s['neg']/s['total']*100))

## Step 6: B2 — Three-Position Interpretive Distribution

Maps Polarity x Sentiment Label to three interpretive positions:
- **Cultural appreciation**: positive + praise/love/support/recommendation
- **Commercial suspicion**: negative + criticism/anger
- **Entertainment prioritization**: remainder (mixed affect, expectation, etc.)

In [ ]:
total_cultural = len(cultural_idx)
polarity_label = Counter()
for i in cultural_idx:
    polarity_label[polarities[i] + '/' + labels_list[i]] += 1

print(f'=== Cultural Comments: Polarity x Label (N={total_cultural}) ===\n')
for k, v in sorted(polarity_label.items(), key=lambda x: -x[1]):
    print('{:<20} {:>6} ({:.1f}%)'.format(k, v, v / total_cultural * 100))

pos_labels = ['赞美', '喜爱', '感动', '支持', '推荐']
neg_labels = ['批评', '愤怒']

appreciation = sum(v for k, v in polarity_label.items()
                   if k.startswith('积极') and k.split('/')[1] in pos_labels)
suspicion = sum(v for k, v in polarity_label.items()
                if k.startswith('消极') and k.split('/')[1] in neg_labels)
entertainment = total_cultural - appreciation - suspicion

print(f'\n=== Three-Position Interpretive Distribution ===')
print(f'Cultural appreciation:  {appreciation:>6} ({appreciation/total_cultural*100:.1f}%)')
print(f'Commercial suspicion:   {suspicion:>6} ({suspicion/total_cultural*100:.1f}%)')
print(f'Entertainment priorit.: {entertainment:>6} ({entertainment/total_cultural*100:.1f}%)')

## Summary of Results

These results are reported in the paper (Section 5, Findings B2):

| Metric | Value |
|--------|-------|
| Cultural-topic comments (N) | 95,220 |
| Commercial-topic comments (N) | 212,795 |
| Overlap | 62,984 |
| Cultural appreciation | 19.9% (n = 18,923) |
| Commercial suspicion | 45.7% (n = 43,485) |
| Entertainment prioritization | 34.5% (n = 32,812) |
| Music-topic positive sentiment | 72.0% |
| Wuxia-topic positive/negative | 45.4% / 48.5% |